# GFP Parameter Experiments — HI-Small

The paper (Altman et al. 2023, Appendix D) fixes one GFP configuration for **all** datasets:
scatter-gather window 6h, everything else 24h, simple cycles up to length 10, bins [2,3,5].
But measuring HI-Small's own 370 laundering attempts (`HI-Small_Patterns.txt`) shows those
defaults are questionable **for this dataset**:

| Measurement on HI-Small patterns | Result | Implication |
|---|---|---|
| Attempt duration | only **21% finish within 24h**; 90% within 120h | 24h windows see a fraction of most patterns |
| Gap between consecutive txns in an attempt | 52% ≤ 24h, **84% ≤ 48h** | a 24h window often cannot connect adjacent hops |
| Cycle sizes | 2–12 accounts; **37% longer than 6**; only 5/54 exceed 10 | `lc-cycle_len=6` misses a third of cycles |
| Fan degrees | median 7.5, p75 = 12, max 16 | bins [2,3,5] dump ~75% of fan patterns into one bin |

## Variants (each changes ONE factor vs the paper config V0)

| Variant | Change | Hypothesis |
|---|---|---|
| **V0 paper** | — (already inside `edge_features.csv` from `Data_preparation.ipynb`) | reference |
| **V1 win48** | sg 12h, cycles/vstats 48h | capture 84% of hop gaps instead of 52% |
| **V2 win120** | sg 24h, cycles/vstats 120h | cover 90% of full attempt durations |
| **V3 lc10** | lc-cycle length 10 | the paper's true cycle length; covers 49/54 observed cycles |
| **V4 rich** | + fan/degree histograms, bins [2,4,8,13] + min/max/median vertex stats | data-driven bins from fan-degree quartiles (2 / 7.5 / 12, max 16) |

Each variant is saved as a row-aligned float32 block in `Data/gfp_variants/<name>.npy`
(+ column names in `<name>_cols.json`), so model notebooks can swap GFP blocks freely.
GBT and GNN comparisons across variants happen in the modeling notebooks.

## 0. Setup

In [1]:
import gc
import json
import os
import pickle
import subprocess
import time
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

class Timer:
    def __init__(self, label): self.label = label
    def __enter__(self): self.t = time.time()
    def __exit__(self, *a): print(f'  [{self.label}] done in {time.time()-self.t:.1f}s')

os.makedirs('Data/gfp_variants', exist_ok=True)
print('ready')

ready


## 1. Rebuild the GFP Input Matrix

Reuses the outputs of `Data_preparation.ipynb` (same truncation, same time order, same
account indexing), so every variant is row-aligned with `edge_features.csv`.

In [2]:
# Only the columns needed to rebuild the GFP input (keeps memory low)
edge_features = pd.read_csv(
    'Data/edge_features.csv',
    usecols=['src_account', 'dst_account', 'Timestamp', 'label', 'Amount_Log'],
    low_memory=False)
# format='mixed': midnight timestamps were written to CSV without a time part
edge_features['Timestamp'] = pd.to_datetime(edge_features['Timestamp'], format='mixed')

with open('Data/account_to_idx.pkl', 'rb') as f:
    account_to_idx = pickle.load(f)

# GFP input layout: [txn_id, src_idx, dst_idx, seconds, amount_usd]
# Amount_USD is recovered from the stored log1p value
seconds = edge_features['Timestamp'].astype('int64') // 10**9
gfp_input = np.column_stack([
    np.arange(len(edge_features)),
    edge_features['src_account'].map(account_to_idx),
    edge_features['dst_account'].map(account_to_idx),
    seconds - seconds.min(),
    np.expm1(edge_features['Amount_Log']),
]).astype(np.float64)

np.save('Data/_gfp_input.npy', gfp_input)
labels = edge_features['label'].to_numpy()          # kept for the signal summary
n_rows = len(edge_features)
del gfp_input, edge_features
gc.collect()

print(f'GFP input saved: {n_rows:,} edges, {labels.sum():,} laundering')

GFP input saved: 5,077,237 edges, 4,522 laundering


## 2. Variant Definitions

`PAPER_PARAMS` is the V0 reference (identical to `Data_preparation.ipynb`); each variant
overrides only the parameters it tests.

In [3]:
PAPER_PARAMS = {
    'num_threads': 8,
    'time_window': 24 * 60 * 60,

    # vertex statistics over input columns 3 (timestamp) and 4 (Amount_USD)
    'vertex_stats':       True,
    'vertex_stats_tw':    24 * 60 * 60,
    'vertex_stats_cols':  [3, 4],
    'vertex_stats_feats': [0, 1, 2, 3, 4, 8, 9, 10],
    # feat ids: 0 fan, 1 degree, 2 ratio, 3 avg, 4 sum, 5 min, 6 max,
    #           7 median, 8 var, 9 skew, 10 kurtosis

    'fan':    False, 'fan_bins':    [2, 5, 10],
    'degree': False, 'degree_bins': [2, 5, 10],

    'scatter-gather':      True,
    'scatter-gather_tw':   6 * 60 * 60,
    'scatter-gather_bins': [2, 3, 5],

    'temp-cycle':      True,
    'temp-cycle_tw':   24 * 60 * 60,
    'temp-cycle_bins': [2, 3, 5],

    'lc-cycle':      True,
    'lc-cycle_tw':   24 * 60 * 60,
    'lc-cycle_bins': [2, 3, 5],
    'lc-cycle_len':  6,
}

H = 60 * 60  # one hour in seconds

# NOTE (verified experimentally): snapml silently caps the per-pattern windows
# by the global 'time_window'. A first run without raising it produced cycle
# features identical to the 24h reference — so the long-window variants must
# raise 'time_window' together with the per-pattern windows.
VARIANTS = {
    # longer windows: capture multi-day patterns (see evidence table above)
    'win48':  {'time_window': 48*H,
               'scatter-gather_tw': 12*H, 'temp-cycle_tw': 48*H,
               'lc-cycle_tw': 48*H, 'vertex_stats_tw': 48*H},
    'win120': {'time_window': 120*H,
               'scatter-gather_tw': 24*H, 'temp-cycle_tw': 120*H,
               'lc-cycle_tw': 120*H, 'vertex_stats_tw': 120*H},

    # longer simple cycles: the paper's own setting
    'lc10':   {'lc-cycle_len': 10},

    # more granular features: fan/degree histograms with data-driven bins
    # + the full vertex statistic set (adds min, max, median).
    # fan_tw/degree_tw set explicitly: the library default is 12h, not 24h.
    'rich':   {'fan': True, 'fan_bins': [2, 4, 8, 13], 'fan_tw': 24*H,
               'degree': True, 'degree_bins': [2, 4, 8, 13], 'degree_tw': 24*H,
               'vertex_stats_feats': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]},
}

for name, overrides in VARIANTS.items():
    print(f'{name:<7}: {overrides}')

win48  : {'time_window': 172800, 'scatter-gather_tw': 43200, 'temp-cycle_tw': 172800, 'lc-cycle_tw': 172800, 'vertex_stats_tw': 172800}
win120 : {'time_window': 432000, 'scatter-gather_tw': 86400, 'temp-cycle_tw': 432000, 'lc-cycle_tw': 432000, 'vertex_stats_tw': 432000}
lc10   : {'lc-cycle_len': 10}
rich   : {'fan': True, 'fan_bins': [2, 4, 8, 13], 'fan_tw': 86400, 'degree': True, 'degree_bins': [2, 4, 8, 13], 'degree_tw': 86400, 'vertex_stats_feats': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]}


## 3. Output Column Names

Reconstructs the column names GFP emits for a given parameter set (pattern histograms
first, then vertex statistics — the order is fixed by the library).

In [4]:
# feat id -> name, in GFP's fixed order
FEAT_NAMES = ['fan', 'deg', 'ratio', 'avg', 'sum', 'min', 'max',
              'median', 'var', 'skew', 'kurtosis']


def build_colnames(params):
    cols = []

    # 1) pattern histograms — one column per bin, last bin open-ended
    for pattern in ['fan', 'degree', 'scatter-gather', 'temp-cycle', 'lc-cycle']:
        if not params.get(pattern):
            continue
        bins = params[f'{pattern}_bins']
        edges_ = [f'{lo}-{hi}' for lo, hi in zip(bins, bins[1:])] + [f'{bins[-1]}-inf']
        if pattern in ('fan', 'degree'):            # these have an in and an out histogram
            for side in ['in', 'out']:
                cols += [f'{pattern}_{side}_bins_{e}' for e in edges_]
        else:
            cols += [f'{pattern}_bins_{e}' for e in edges_]

    # 2) vertex statistics — 4 groups: source/dest x out/in
    feats = params['vertex_stats_feats']
    for who in ['source', 'dest']:
        for direction in ['out', 'in']:
            # topology values (fan, deg, ratio) come first ...
            cols += [f'{who}_{FEAT_NAMES[k]}_{direction}' for k in feats if k <= 2]
            # ... then the statistics of each input column (ts, then amount)
            for label in ['ts', 'amt']:
                cols += [f'{who}_{FEAT_NAMES[k]}_{label}_{direction}'
                         for k in feats if k >= 3]
    return cols


# sanity: V0 must reproduce the 61 columns of Data_preparation.ipynb
assert len(build_colnames(PAPER_PARAMS)) == 61
for name, overrides in VARIANTS.items():
    n = len(build_colnames({**PAPER_PARAMS, **overrides}))
    print(f'{name:<7}: {n} feature columns')

win48  : 61 feature columns
win120 : 61 feature columns
lc10   : 61 feature columns
rich   : 101 feature columns


## 4. Run All Variants

Same WSL streaming bridge as `Data_preparation.ipynb` (batch 128, causal). Each variant's
feature block is verified for bounded leakage and saved to `Data/gfp_variants/`.

In [5]:
WSL_DIR = '/mnt/c/Users/User/Desktop/Sapienza/Master Thesis/gnn_for_fraudulent_patterns'
GFP_BATCH = 128


def run_variant(name, overrides):
    params = {**PAPER_PARAMS, **overrides}
    cols = build_colnames(params)

    # hand the parameters to the WSL runner
    with open('Data/_gfp_params.json', 'w') as f:
        json.dump(params, f)
    cmd = (f'~/gfp_env/bin/python "{WSL_DIR}/run_gfp_wsl.py" '
           f'"{WSL_DIR}/Data/_gfp_input.npy" "{WSL_DIR}/Data/_gfp_params.json" '
           f'"{WSL_DIR}/Data/_gfp_output.npy" {GFP_BATCH}')
    result = subprocess.run(['wsl', 'bash', '-c', cmd], capture_output=True, text=True)
    assert result.returncode == 0, f'{name} failed:\n{result.stderr[-2000:]}'

    # keep only the feature columns (first 5 are the input passed through)
    out = np.load('Data/_gfp_output.npy')
    assert out.shape == (n_rows, 5 + len(cols)), f'{name}: unexpected shape {out.shape}'
    feats = out[:, 5:]
    del out
    gc.collect()

    np.save(f'Data/gfp_variants/{name}.npy', feats)
    with open(f'Data/gfp_variants/{name}_cols.json', 'w') as f:
        json.dump(cols, f, indent=1)
    print(f'{name:<7}: saved {feats.shape}')
    del feats
    gc.collect()


for name, overrides in VARIANTS.items():
    with Timer(name):
        run_variant(name, overrides)

# clean up the temp bridge files
for tmp in ['Data/_gfp_input.npy', 'Data/_gfp_params.json', 'Data/_gfp_output.npy']:
    if os.path.exists(tmp):
        os.remove(tmp)
print('\nAll variants done')

win48  : saved (5077237, 61)
  [win48] done in 445.4s


win120 : saved (5077237, 61)
  [win120] done in 600.9s


lc10   : saved (5077237, 61)


  [lc10] done in 282.8s


rich   : saved (5077237, 101)


  [rich] done in 889.1s



All variants done


## 5. Quick Signal Summary

Correlation with the laundering label per variant — a first sanity look before the real
GBT / GNN comparison in the modeling notebooks (V0 numbers come from `edge_features.csv`).

In [6]:
for name in VARIANTS:
    feats = np.load(f'Data/gfp_variants/{name}.npy')
    cols = json.load(open(f'Data/gfp_variants/{name}_cols.json'))

    # Pearson correlation of every feature column with the binary label
    x = np.nan_to_num(feats)
    xc = x - x.mean(axis=0)
    yc = labels - labels.mean()
    corr = (xc * yc[:, None]).sum(0) / (
        np.sqrt((xc**2).sum(0) * (yc**2).sum()) + 1e-12)

    top = np.argsort(-np.abs(corr))[:5]
    print(f'=== {name} — top 5 |corr| with label '
          f'({(np.abs(corr) > 0.01).sum()}/{len(cols)} above 0.01) ===')
    for i in top:
        print(f'  {cols[i]:<32} {corr[i]:+.4f}')
    del feats, x, xc
    gc.collect()

=== win48 — top 5 |corr| with label (12/61 above 0.01) ===
  lc-cycle_bins_3-5                +0.0817
  temp-cycle_bins_3-5              +0.0781
  temp-cycle_bins_2-3              +0.0640
  lc-cycle_bins_2-3                +0.0640
  lc-cycle_bins_5-inf              +0.0605


=== win120 — top 5 |corr| with label (13/61 above 0.01) ===
  temp-cycle_bins_3-5              +0.0970
  lc-cycle_bins_3-5                +0.0903
  temp-cycle_bins_5-inf            +0.0840
  lc-cycle_bins_5-inf              +0.0793
  temp-cycle_bins_2-3              +0.0618


=== lc10 — top 5 |corr| with label (11/61 above 0.01) ===
  temp-cycle_bins_2-3              +0.0635
  lc-cycle_bins_2-3                +0.0635
  lc-cycle_bins_3-5                +0.0589
  temp-cycle_bins_3-5              +0.0434
  lc-cycle_bins_5-inf              +0.0252


=== rich — top 5 |corr| with label (20/101 above 0.01) ===
  lc-cycle_bins_2-3                +0.0635
  temp-cycle_bins_2-3              +0.0635
  lc-cycle_bins_3-5                +0.0589
  temp-cycle_bins_3-5              +0.0434
  lc-cycle_bins_5-inf              +0.0285


## How to Use the Variants in Model Notebooks

```python
meta  = json.load(open('Data/feature_meta.json'))
base  = edge_features[meta['BASE_EDGE_COLS']].to_numpy()          # 20 baseline features
gfp   = np.load('Data/gfp_variants/win48.npy')                    # any variant (V0 = meta['GFP_FEAT_COLS'] inside edge_features.csv)
X     = np.hstack([base, gfp])                                    # full edge features
```

Normalize variant vertex-stat columns exactly like `Data_preparation.ipynb` Section 6
(log1p → clip at train p1/p99 → StandardScaler, fit on the first 60% of rows only).

**Ablation grid for every model** (defined in `feature_meta.json`):
A. node features + structure only · B. + baseline edges · C. + GFP · D. full.